# The Laplace approximation, honestly

Everything happens in unconstrained coordinates (review B13): the mode is found there, the
Hessian is the *real* curvature at the mode — never an optimizer's BFGS approximation, the bug
the parent shipped — and draws are mapped back through the transforms. If the mode search does
not converge, or the Hessian is not positive definite, you get an `Unverified` with the
diagnostics, not a posterior. `nonfinite_draw_frac` is reported on every result.

In [ ]:
import numpy as np

from axiom.core import D, Data, Gather, Likelihood, ModelSpec, Param, Prior, Unverified, dimensionless, is_failure, value
from axiom.infer import Posterior, find_mode, hessian_at, laplace

## A hierarchical model — the case that broke the parent

Three units with their own intercepts under a shared hyperprior.

In [ ]:
unit = Data(name="unit", dimension=dimensionless())
y = Data(name="y", dimension=D.outcome)
a_mean = Param(name="a_mean", dimension=D.outcome, prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 5.0}))
a_sd = Param(name="a_sd", dimension=D.outcome, prior=Prior(family="halfnormal", hyper={"sigma": 2.0}))
alpha = Param(name="alpha", dimension=D.outcome, shape=(3,), prior=Prior(family="normal", hyper={"mu": "a_mean", "sigma": "a_sd"}))
sigma = Param(name="sigma", dimension=D.outcome, prior=Prior(family="halfnormal", hyper={"sigma": 2.0}))
model = ModelSpec(name="hier", mean=Gather(source=alpha, index=unit), outcome=y, likelihood=Likelihood(family="normal", scale="sigma"), parameters=(a_mean, a_sd, alpha, sigma))
rng = np.random.default_rng(1)
idx = np.repeat([0, 1, 2], 30)
data = {"unit": idx, "y": np.array([1.0, 2.5, 4.0])[idx] + rng.normal(0, 0.7, 90)}

In [ ]:
mode = find_mode(model, data)
print(mode if is_failure(mode) else (mode.method, mode.converged, {k: np.round(v, 3) for k, v in mode.theta.items()}))
post = laplace(model, data, draws=4000, seed=0)
if isinstance(post, Posterior):
    prov = post.provenance
    print({k: prov[k] for k in ("hessian_pd", "nonfinite_draw_frac", "converged", "min_eigenvalue")})
    print(post.summary("a_sd"), post.draws("alpha").shape)
else:
    print(post)

## The Hessian is real curvature

`hessian_at` evaluates the Hessian of the negative log density at a point — by jax when
installed, else by unit-scaled finite differences. Compare the two.

In [ ]:
from axiom.core import jax_available, unconstrain

z = unconstrain(model, {"a_mean": 2.5, "a_sd": 1.0, "alpha": np.array([1.0, 2.5, 4.0]), "sigma": 0.7})
H_fd = hessian_at(model, data, z, derivatives="finite_difference")
print(np.round(np.linalg.eigvalsh(H_fd), 2))
if jax_available():
    H_jax = hessian_at(model, data, z, derivatives="jax")
    print("max |H_jax - H_fd| / |H_jax|:", float(np.max(np.abs(H_jax - H_fd)) / np.max(np.abs(H_jax))))

## When it cannot be trusted

A product-form mean `a · b` has a saddle at the origin. The mode search restarts, and if it
cannot reach a positive-definite mode it says so.

In [ ]:
a = Param(name="a", dimension=D.outcome, prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 10.0}))
b = Param(name="b", dimension=dimensionless(), prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 10.0}))
from axiom.core import Mul

prod = ModelSpec(name="product", mean=Mul(factors=(a, b)), outcome=y, likelihood=Likelihood(family="normal", scale="sigma"), parameters=(a, b, sigma))
pdata = {"y": rng.normal(3.0, 0.5, 40)}
res = laplace(prod, pdata, draws=500, seed=0)
print(type(res).__name__, res.provenance.get("n_restarts") if isinstance(res, Posterior) else res.reason[:120])
flat = laplace(prod, pdata, draws=500, seed=0, allow_unverified=True)
print(type(flat).__name__)